# System Resource Monitor

This notebook samples Windows system resources on a fixed interval for a fixed total
duration and writes the results to two Excel workbooks: a dashboard that always holds
the latest snapshot and a log that grows for the full run. Every row on every sheet of
both files carries the sample timestamp.

## Config

This is the only cell you adjust. It holds the sample interval, the total run duration,
and the two output file paths. Every later section reads from these values, so changing
them here changes the whole run. The defaults are a 5 minute interval over 24 hours,
which is 288 samples.

In [3]:
from datetime import timedelta
from pathlib import Path

# How long to wait between samples.
SAMPLE_INTERVAL = timedelta(minutes=5)

# How long the whole run lasts. After this elapses the loop stops.
TOTAL_DURATION = timedelta(hours=24)

# Output file paths. Do not change these names.
DASHBOARD_PATH = Path("outputs") / "dashboard" / "dashboard.xlsx"
LOG_PATH = Path("outputs") / "log" / "log.xlsx"

# Make sure the output folders exist.
DASHBOARD_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# Derived: total number of samples this run will take.
SAMPLE_COUNT = int(TOTAL_DURATION / SAMPLE_INTERVAL)

print("Sample interval :", SAMPLE_INTERVAL)
print("Total duration  :", TOTAL_DURATION)
print("Planned samples :", SAMPLE_COUNT)
print("Dashboard file  :", DASHBOARD_PATH)
print("Log file        :", LOG_PATH)

Sample interval : 0:05:00
Total duration  : 1 day, 0:00:00
Planned samples : 288
Dashboard file  : outputs\dashboard\dashboard.xlsx
Log file        : outputs\log\log.xlsx


## Dependencies setup

This section makes the notebook run on a fresh machine that has nothing installed beyond
Python itself. It checks for each package the notebook needs and installs only the ones
that are missing, so running it twice does no harm. The packages are `psutil` for reading
system resources and `openpyxl` for writing the Excel files.

One prerequisite cannot be handled here: Python 3 must already be installed, because the
notebook needs Python to run at all. If you are on a machine with no Python, install
Python 3 from python.org first, then open this notebook. Everything else is installed
automatically by the cell below.

In [4]:
import importlib
import subprocess
import sys

# Import name mapped to the name pip installs it under.
REQUIRED_PACKAGES = {
    "psutil": "psutil",
    "openpyxl": "openpyxl",
}


def ensure_packages(packages):
    """Report each package as installed or not installed, then install the missing."""
    print("Python version:", sys.version.split()[0])
    print("-" * 40)

    missing = []
    for import_name, pip_name in packages.items():
        try:
            importlib.import_module(import_name)
            print("INSTALLED      :", pip_name)
        except ImportError:
            print("NOT INSTALLED  :", pip_name)
            missing.append((import_name, pip_name))

    print("-" * 40)
    if not missing:
        print("Nothing to install. All required packages are ready.")
        return

    for import_name, pip_name in missing:
        print("Installing", pip_name, "...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        importlib.import_module(import_name)
        print("Installed      :", pip_name)

    print("-" * 40)
    print("All required packages are ready.")


ensure_packages(REQUIRED_PACKAGES)


Python version: 3.12.7
----------------------------------------
INSTALLED      : psutil
INSTALLED      : openpyxl
----------------------------------------
Nothing to install. All required packages are ready.


## CPU usage

This section reads CPU load. It returns the overall busy percentage and the busy
percentage of each logical core. The reading uses a one second measurement window
(`interval=1`) so the value reflects real activity rather than the zero you get from an
instant call. The overall figure is the mean of the per-core figures, so the two always
agree. The result is a flat dictionary of named fields plus the sample timestamp, which
is the shape every system-metrics collector in this notebook returns.

In [5]:
import psutil
from datetime import datetime


def collect_cpu(timestamp):
    """Return overall and per-core CPU usage for one sample, in plain labels."""
    per_core = psutil.cpu_percent(interval=1, percpu=True)
    overall = round(sum(per_core) / len(per_core), 1) if per_core else 0.0

    row = {
        "Time": timestamp,
        "Overall CPU Usage (%)": overall,
        "Number of CPU Cores": len(per_core),
    }
    for index, value in enumerate(per_core):
        # Cores numbered from 1 so the labels read naturally.
        row["Core {} Usage (%)".format(index + 1)] = value
    return row


# Quick check.
collect_cpu(datetime.now())


{'Time': datetime.datetime(2026, 6, 10, 14, 21, 46, 817487),
 'Overall CPU Usage (%)': 33.8,
 'Number of CPU Cores': 8,
 'Core 1 Usage (%)': 32.3,
 'Core 2 Usage (%)': 24.6,
 'Core 3 Usage (%)': 73.4,
 'Core 4 Usage (%)': 39.1,
 'Core 5 Usage (%)': 27.7,
 'Core 6 Usage (%)': 25.0,
 'Core 7 Usage (%)': 21.9,
 'Core 8 Usage (%)': 26.6}

## CPU temperature

This section reads the CPU thermal sensor. Windows does not expose CPU temperature
through psutil, so the reading is pulled from the WMI thermal zone using a short
PowerShell call. WMI reports the value in tenths of a Kelvin, which this code converts to
degrees Celsius. Many laptops block this sensor or return access denied. When that
happens the field is recorded as `Unavailable` for that sample rather than letting the
error stop the run, so the timestamp and the rest of the sample are still captured.

In [6]:
import subprocess
from datetime import datetime


def _run_powershell(command, timeout=15):
    """Run a PowerShell command and return its text output, or None on failure."""
    try:
        result = subprocess.run(
            ["powershell", "-NoProfile", "-NonInteractive", "-Command", command],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        pass
    return None


def collect_temperature(timestamp):
    """Return the CPU temperature in Celsius, or Unavailable if the sensor is blocked."""
    command = (
        "Get-CimInstance -Namespace root/wmi "
        "-ClassName MSAcpi_ThermalZoneTemperature -ErrorAction Stop "
        "| Select-Object -ExpandProperty CurrentTemperature"
    )
    output = _run_powershell(command)

    temperature = "Unavailable"
    if output:
        readings = []
        for line in output.splitlines():
            line = line.strip()
            if line.isdigit():
                # WMI reports tenths of a Kelvin. Convert to Celsius.
                readings.append(int(line) / 10.0 - 273.15)
        if readings:
            temperature = round(sum(readings) / len(readings), 1)

    return {
        "Time": timestamp,
        "CPU Temperature (°C)": temperature,
    }


# Quick check.
collect_temperature(datetime.now())

{'Time': datetime.datetime(2026, 6, 10, 14, 21, 47, 835787),
 'CPU Temperature (°C)': 'Unavailable'}

## RAM

This section reads system memory. It reports the total memory fitted, how much is in use,
how much is still available, and the used percentage. The raw figures from the system are
in bytes, which are hard to read, so they are converted to gigabytes (GB) and rounded.
Available memory is the amount that can be handed to programs without swapping, which is
the number most people care about when asking how much memory is free.

In [7]:
import psutil
from datetime import datetime

BYTES_PER_GB = 1024 ** 3


def to_gb(value):
    """Convert a byte count to gigabytes, rounded to two decimals."""
    return round(value / BYTES_PER_GB, 2)


def collect_ram(timestamp):
    """Return total, used, available memory in GB and the used percentage."""
    memory = psutil.virtual_memory()
    return {
        "Time": timestamp,
        "Total Memory (GB)": to_gb(memory.total),
        "Used Memory (GB)": to_gb(memory.used),
        "Available Memory (GB)": to_gb(memory.available),
        "Memory Usage (%)": memory.percent,
    }


# Quick check.
collect_ram(datetime.now())

{'Time': datetime.datetime(2026, 6, 10, 14, 26, 8, 906675),
 'Total Memory (GB)': 7.71,
 'Used Memory (GB)': 6.31,
 'Available Memory (GB)': 1.4,
 'Memory Usage (%)': 81.8}